# Introduction
**In this notebook i will fine-tune a pre-trained BERT model to automatically detect whether an online comment is toxic or not. This is a real-world binary text classification problem solved using Hugging Face Transformers and PyTorch.**


# Basic Imports

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader
!pip install transformers

In [2]:
from transformers import BertTokenizer , BertForSequenceClassification

# Loading Training Data

In [3]:
df = pd.read_csv("/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip")

# Data Exploration And Balancing
- Analyzing the class distribution — the dataset is heavily imbalanced with ~90% non-toxic comments. Sample 2,000 examples from each class to create a balanced dataset of 4,000 comments.


In [4]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
# Checking Shape
len(df)

159571

In [6]:
df.shape

(159571, 8)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [8]:
df['toxic'].value_counts()

toxic
0    144277
1     15294
Name: count, dtype: int64

In [9]:
toxic = df[df['toxic'] == 1].sample(2000, random_state=42)
non_toxic = df[df['toxic'] == 0].sample(2000, random_state=42)
df_balanced = pd.concat([toxic, non_toxic]).sample(frac=1, random_state=42).reset_index(drop=True)

In [10]:
df_balanced['toxic'].value_counts()

toxic
1    2000
0    2000
Name: count, dtype: int64

# Tokenizing the Data
**Loading the BERT tokenizer and converting raw comment text into numerical token IDs and attention masks that BERT understands. Use `max_length=128`, `padding='max_length'`, and `truncation=True`.**


In [11]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Tokenizing The Balanced Dataset

In [12]:
encodings = tokenizer(df_balanced['comment_text'].tolist(), padding='max_length', truncation=True, max_length=128)

# Creating PyTorch Dataset Class
- Building a custom `ToxicDataset` class that wraps the tokenized encodings and labels, teaching PyTorch how to access individual examples during training.


In [13]:
class Toxicdataset(Dataset):
    def __init__ (self,encodings,labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self , idx):
        return {
        "input_ids" : torch.tensor(self.encodings["input_ids"][idx]),
        "attention_mask" : torch.tensor(self.encodings["attention_mask"][idx]),
        "labels" : torch.tensor(self.labels[idx])}
    

# Splitting Data

In [14]:
from sklearn.model_selection import train_test_split

train_df , test_df = train_test_split(df_balanced , test_size = 0.2 , random_state = 42)

## Splitting Again
- Since the whole dataset was encoded before we have to tokenize train and test seperately

In [15]:
train_encoded = tokenizer(train_df['comment_text'].tolist(), padding='max_length', truncation=True, max_length=128 )
test_encoded = tokenizer(test_df['comment_text'].tolist(), padding='max_length', truncation=True, max_length=128 )

# Creating Dataset Object

In [16]:
train_dataset = Toxicdataset(train_encoded, train_df['toxic'].tolist())
test_dataset = Toxicdataset(test_encoded, test_df['toxic'].tolist())

# Loading Datasets Into DataLoader

In [17]:
train_dataloader = DataLoader(train_dataset , batch_size = 16 , shuffle = True)
test_dataloader = DataLoader(test_dataset , batch_size = 16 , shuffle = False)

# Loading The Model
- Loading `BertForSequenceClassification` with `num_labels=2` from Hugging Face. Move the model to GPU if available, otherwise CPU.


In [18]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased' , num_labels = 2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Setting The Optimizer

In [19]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters() , lr = 2e-5)

# Model Training
- Running the training loop for 1 epoch — feeding batches through the model, computing loss, backpropagating gradients, and updating weights. Printing average loss at the end.


In [20]:
epochs = 1
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        output = model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            labels = labels
        )
        loss = output.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss/ len(train_dataloader)
    print(f"Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}")

Epoch 1/1 - Loss: 0.2867


# Predict Function
- Writing a `predict(text)` function that tokenizes raw input text and returns whether the comment is **Toxic** or **Not Toxic**.


In [21]:
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors = 'pt',
        max_length = 128,
        padding = 'max_length',
        truncation = True
    ).to(device)
    with torch.no_grad():
        output = model(**inputs)
    prediction = torch.argmax(output.logits, dim=1).item()
    
    labels = {0: "Not Toxic", 1: "Toxic"}
    print(labels[prediction])

# Testing

In [22]:
predict("You are such a wonderful person, thank you for your help!")
predict("I hate you, you're absolutely disgusting and stupid.")
predict("This is a great discussion, I learned a lot.")

Not Toxic
Toxic
Not Toxic


# Conclusion
- In this project we built a toxic comment detector by fine-tuning BERT on the Jigsaw dataset. A key challenge was class imbalance — 90% of the data was non-toxic — which we solved by sampling 2,000 examples from each class before training. The model successfully learned to distinguish toxic from non-toxic comments in just 1 epoch with a training loss of 0.30, correctly classifying test comments like "I hate you, you're absolutely disgusting" as Toxic and "This is a great discussion" as Not Toxic. This shows that even with a small balanced sample, a pre-trained BERT model can be quickly adapted to detect harmful content — a problem faced by real platforms like YouTube, Reddit, and Twitter every day.